# Qiskit Fall Fest 2026: Industry Challenge I7 — Noise-Aware Engineering
## Evidence-Driven Quantum Error Mitigation Benchmark

**Participant Role:** Quantum Software Engineer  
**Problem Statement:** Quantum cloud teams need reliable execution results on noisy physical hardware. This notebook provides an evidence-based, reproducible pipeline demonstrating:
1. **Ideal Simulation**: Noiseless baseline state sampling.
2. **Target Backend Transpilation & Profiling**: Gate counts, 2Q entangling operations, and circuit depth.
3. **Realistic Noise Simulation**: Qiskit Aer depolarizing gate noise and asymmetric readout assignment errors across Low, Medium, and High regimes.
4. **Error Mitigation (ZNE)**: Unitary gate folding ($G \to G G^\dagger G$) noise scaling and polynomial/Richardson zero-noise extrapolation.
5. **Quantitative Evaluation**: Total Variation Distance (TVD), Hellinger Fidelity ($F_{cl}$), and Error Reduction Gain (%).
6. **Classical Baseline**: CPU-based exact statevector evaluation vs sample-based quantum execution.

> **Important Scientific Honesty Guidelines:**
> - Mitigation reduces the *effect* of noise on measured expectation values/distributions; it does **not** physically eliminate hardware decoherence.
> - No quantum advantage is claimed. Classical simulators efficiently handle small problem sizes ($N \le 20$).
> - All figures and tables are computed from actual Qiskit runtime executions.

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import qiskit
import qiskit_aer

print(f"Qiskit Version: {qiskit.__version__}")
print(f"Qiskit Aer Version: {qiskit_aer.__version__}")

### 1. Execute Benchmark Suite
Run the complete modular pipeline across all test circuits (Bell State, GHZ State, QAOA Ansatz) and noise profiles (Low, Medium, High).

In [ ]:
from src.experiments.run_experiments import run_full_suite

results = run_full_suite()
df_summary = pd.read_csv('results/processed/benchmark_summary.csv')
df_summary

### 2. Quantitative Error & Fidelity Metrics Analysis
Inspect the measured Hellinger fidelity gains and Total Variation Distance reductions.

In [ ]:
for c in df_summary['circuit_name'].unique():
    sub = df_summary[df_summary['circuit_name'] == c]
    print(f"\nCircuit: {c}")
    for _, row in sub.iterrows():
        print(f"  [{row['noise_level'].upper():6s}] Raw Fid: {row['fidelity_noisy']:.4f} -> Mitigated Fid: {row['fidelity_mitigated']:.4f} (Gain: {row['fidelity_improvement_pct']:+.2f}%)")

### 3. Visualizations & Scientific Findings
Examine generated publication-grade figures.

In [ ]:
from IPython.display import Image, display

fig_files = [
    'results/figures/dist_comparison_bell_state_2q.png',
    'results/figures/dist_comparison_ghz_state_3q.png',
    'results/figures/dist_comparison_qaoa_ansatz_3q_p2.png',
    'results/figures/fidelity_mitigation_benchmark.png',
    'results/figures/mitigation_gain_by_noise_regime.png',
    'results/figures/transpilation_structural_profile.png',
]

for fig in fig_files:
    if os.path.exists(fig):
        print(f"Displaying: {fig}")
        display(Image(filename=fig))

### 4. Classical Baseline Comparison
Comparison of CPU exact statevector simulation runtime against quantum sample execution.

In [ ]:
print(df_summary[['circuit_name', 'noise_level', 'qubits', 'shots', 'classical_time_sec']])